In [1]:
from transformers import pipeline
import os
import json
from datetime import datetime
import gradio as gr
import logging

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.makedirs("logs", exist_ok=True)
logging.basicConfig(filename="logs/transcribe.log", level=logging.INFO, format="%(message)s")

In [3]:
asr = pipeline(task="automatic-speech-recognition",
               model="distil-whisper/distil-small.en",
               model_kwargs={"cache_dir": "models"})

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 14244.90it/s]


In [4]:
asr.feature_extractor.sampling_rate

16000

In [ ]:
def inference_hardware():
    on_gpu = asr.device.type == "cuda"
    in_colab = "COLAB_RELEASE_TAG" in os.environ
    if on_gpu and in_colab:
        return "colab gpu"
    if on_gpu:
        return "local gpu"
    return "cpu"

def log_transcription(filepath, text, latency_ms):
    if filepath is not None:
        audio_dir = os.path.abspath("logs/audio")
        src = os.path.abspath(filepath)
        if src.startswith(audio_dir + os.sep):
            stem = "".join(c for c in text.strip() if c.isalnum() or c in " ,.'-")[:80].strip() or "audio"
            ext = os.path.splitext(src)[1]
            dest = os.path.join(audio_dir, stem + ext)
            i = 2
            while os.path.exists(dest):
                dest = os.path.join(audio_dir, f"{stem} {i}{ext}")
                i += 1
            os.replace(src, dest)
            os.rmdir(os.path.dirname(src))
            filepath = dest
    filepath_rel = None if filepath is None else os.path.relpath(filepath)
    logging.info(json.dumps({"time": datetime.now().isoformat(), "input": filepath_rel, "output": text, "latency": f'{latency_ms} ms',
        "model": asr.model.name_or_path, "hardware": inference_hardware()}))

def transcribe_speech(filepath):
    latency_ms = None
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        text = ""
    else:
        started = datetime.now()
        output = asr(filepath)
        latency_ms = round((datetime.now() - started).total_seconds() * 1000)
        text = output["text"]
    log_transcription(filepath, text, latency_ms)
    return text

In [6]:
mic_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="microphone",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
)
os.makedirs("logs/audio", exist_ok=True)
mic_transcribe.input_components[0].GRADIO_CACHE = os.path.abspath("logs/audio")

In [7]:
file_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="upload",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
)

In [ ]:
demo = gr.Blocks()

with demo:
    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )

demo.launch(share=True, 
            server_port=int(os.environ.get('PORT1', 7680)))

* Running on local URL:  http://127.0.0.1:7680
* Running on public URL: https://07593b04ae7bc42795.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
demo.close()